### Base de Datos (Database)

#### Objetivos
1. Documentación sobre Spark SQL
2. Crear la Base de Datos "demo"
3. Acceder al "Catalog" en la "Intefaz de Usuario"
4. Comando "SHOW"
5. Comando DESCRIBE(DESC)
6. Mostrar la Base de Datos Actual

In [0]:
CREATE SCHEMA IF NOT EXISTS demo;

In [0]:
SHOW DATABASES;

databaseName
default
demo
information_schema


In [0]:
DESCRIBE DATABASE demo;

database_description_item,database_description_value
Catalog Name,movie_curso_2_databricks
Namespace Name,demo
Comment,
Location,
Owner,wilder.aristizabal898@gmail.com


In [0]:
DESCRIBE DATABASE EXTENDED demo;

database_description_item,database_description_value
Catalog Name,movie_curso_2_databricks
Namespace Name,demo
Comment,
Location,
Owner,wilder.aristizabal898@gmail.com
Properties,
Predictive Optimization,ENABLE (inherited from METASTORE metastore_azure_eastus)


In [0]:
--Consultar en que Base de datos estoy Trabajando.
SELECT current_database();

current_schema()
demo


In [0]:
SHOW TABLES IN demo;

database,tableName,isTemporary
,_sqldf,true


In [0]:
USE demo;

In [0]:
SELECT current_database();

current_schema()
demo


In [0]:
SHOW TABLEs IN default;

database,tableName,isTemporary
,_sqldf,true


### Tablas Administradas(Managed Tables) 

#### Objetivos
1. Crear una "Tabla Adminstrada(Managed Table)" con Python
2. Crear una "Tabla Adminstrada(Managed Table)" con SQL
3. Efecto de eliminar una Tabla Administrada
4. Describir(Describe) la Tabla

In [0]:
%run "../includes/configuration"

In [0]:
%python
results_movie_genre_language = spark.read.parquet(f"{gold_folder_path}/results_movie_genre_language")

In [0]:
%python
results_movie_genre_language.write.format("delta").saveAsTable("demo.results_movie_genre_language_python")

In [0]:
USE demo;
SHOW TABLES;

database,tableName,isTemporary
demo,results_movie_genre_language_python,false
,_sqldf,true


In [0]:
DESCRIBE EXTENDED results_movie_genre_language_python;

col_name,data_type,comment
title,string,null
duration_time,int,null
release_date,date,null
vote_average,double,null
genre_name,string,null
language_name,string,null
created_date,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"duration_time, created_date, vote_average, release_date, genre_name, title, language_name",


In [0]:
CREATE TABLE demo.results_movie_genre_language_sql
AS 
SELECT * 
FROM results_movie_genre_language_python
WHERE genre_name = 'Adventure'

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM results_movie_genre_language_sql;

In [0]:
SELECT current_database();

current_schema()
demo


In [0]:
DESCRIBE EXTENDED results_movie_genre_language_sql;

col_name,data_type,comment
title,string,null
duration_time,int,null
release_date,date,null
vote_average,double,null
genre_name,string,null
language_name,string,null
created_date,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"duration_time, created_date, vote_average, release_date, genre_name, title, language_name",


In [0]:
DROP TABLE IF EXISTS results_movie_genre_language_sql;

In [0]:
SHOW TABLES IN demo;

database,tableName,isTemporary
demo,results_movie_genre_language_python,false
,_sqldf,true


### Tablas Externas(External Tables)

#### Objetivos
1. Crear una "Tabla Externa(External Table)" con Python
2. Crear una "Tabla Externa(External Table)" con SQL
3. Efecto de eliminación de una "Tabla Externa(External Table)"
4. Describir(Describe) la Tabla

In [0]:
%python
results_movie_genre_language.write.format("delta").option("path", f"{gold_folder_path}/results_movie_genre_language_py_2").saveAsTable("demo.results_movie_genre_language_py_2")

In [0]:
DESC EXTENDED demo.results_movie_genre_language_py_2

col_name,data_type,comment
title,string,null
duration_time,int,null
release_date,date,null
vote_average,double,null
genre_name,string,null
language_name,string,null
created_date,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"duration_time, created_date, vote_average, release_date, genre_name, title, language_name",


In [0]:
CREATE TABLE demo.results_movie_genre_language_sql(
    title STRING,
    duration_time INT,
    release_date DATE,
    vote_average FLOAT,
    language_name STRING,
    genre_name STRING,
    crated_date TIMESTAMP
)
USING PARQUET
LOCATION "abfss://gold@stcursodatabricks.dfs.core.windows.net/resuresults_movie_genre_language_ext_sql"

In [0]:
SHOW TABLES IN demo;

database,tableName,isTemporary
demo,results_movie_genre_language_py_2,false
demo,results_movie_genre_language_python,false
demo,results_movie_genre_language_sql,false
,_sqldf,true


In [0]:
INSERT INTO demo.results_movie_genre_language_sql
SELECT * FROM demo.results_movie_genre_language_py_2
WHERE genre_name = 'Adventure'

In [0]:
SELECT count(1)
FROM demo.results_movie_genre_language_sql

count(1)
2538


In [0]:
SHOW TABLES IN demo;

database,tableName,isTemporary
demo,results_movie_genre_language_py_2,false
demo,results_movie_genre_language_python,false
demo,results_movie_genre_language_sql,false
,_sqldf,true


In [0]:
DROP TABLE results_movie_genre_language_sql

In [0]:
SHOW TABLES IN demo;

database,tableName,isTemporary
demo,results_movie_genre_language_py_2,false
demo,results_movie_genre_language_python,false
,_sqldf,true


### Vistas(Views)

#### Objetivos
1. Crear Vista Temporal
2. Crear VistaTemporal Global
3. Crear Vista Permanente

#### 1. Crear Vista Temporal

In [0]:
SELECT current_database();

current_schema()
demo


In [0]:
CREATE OR REPLACE TEMP VIEW v_results_movies_genres_language
AS
SELECT *
FROM demo.results_movie_genre_language_py_2
WHERE genre_name = 'Adventure'

In [0]:
SELECT * FROM v_results_movies_genres_language;

#### 2. Crear VistaTemporal Global

In [0]:
CREATE OR REPLACE GLOBAL TEMP VIEW gv_results_movies_genres_language
AS
SELECT *
FROM demo.results_movie_genre_language_py_2
WHERE genre_name = 'Drama';

In [0]:
SHOW TABLES IN global_temp;

database,tableName,isTemporary
global_temp,gv_movie_genre_language,true
global_temp,gv_results_movies_genres_language,true
,_sqldf,true
,v_results_movies_genres_language,true


In [0]:
SELECT * FROM global_temp.gv_results_movies_genres_language

#### 3. Crear Vista Permanente

In [0]:
CREATE OR REPLACE VIEW pv_results_movies_genres_language
AS
SELECT *
FROM demo.results_movie_genre_language_py_2
WHERE genre_name = 'Comedy';

In [0]:
SHOW TABLES;

database,tableName,isTemporary
demo,pv_results_movies_genres_language,false
demo,results_movie_genre_language_py_2,false
demo,results_movie_genre_language_python,false
,_sqldf,true
,v_results_movies_genres_language,true


In [0]:
SELECT * FROM pv_results_movies_genres_language;